In [156]:
import numpy as np
import matplotlib.pyplot as plt
import imageio

vertices start in a line
update position via monte carlo step
hooke's law
fixed points - boundary condtion
vertices can move in 2d
need to calculate angle between points
determine s by yourself, such that ~50% of the steps are accepted
try doing the move for all points simultaneously, but if it doesnt work, you can do it sequentially

In [ ]:
class System():
    def __init__(self,l_0, N, k, k_b, k_bT):
        # there will be 1 fewer connections than vertices
        self.l = np.full(int(N-1), l_0)
        # arange the points in a line 
        x = np.arange(N) * l_0
        y = np.zeros(int(N))
        self.pos = np.column_stack((x,y))

        # # for testing
        # no = 3
        # self.pos[no] = self.pos[no] + [0,1]
        self.k = k
        self.k_b = k_b
        self.k_bT = k_bT
        # self.plot()


    def plot(self):
        # plt.scatter(self.pos[:,0], self.pos[:,1])
        plt.figure(figsize=(12, 4))
        plt.plot(self.pos[:,0], self.pos[:,1], '.-')
        plt.show

    def calculate_spring_energy(self, verbose=False):
        # spring vectors (i -> i+1)
        diff = self.pos[1:] - self.pos[:-1]

        # spring lengths
        dist = np.linalg.norm(diff, axis=1)

        # extension
        dl = dist - self.l

        # energy per spring
        E_springs = 0.5 * self.k * dl**2

        # per-vertex energy (N vertices)
        E_vertices = np.zeros(self.pos.shape[0])

        # each spring contributes half to each of its two vertices
        E_vertices[:-1] += 0.5 * E_springs
        E_vertices[1:]  += 0.5 * E_springs

        if verbose:
            print("Spring lengths:", dist)
            print("Spring energies:", E_springs)
            print("Vertex energies:", E_vertices)

        return E_vertices

    def calculate_bending_energy(self, verbose=False):
        # neighbors
        prev = np.roll(self.pos, 1, axis=0)
        next = np.roll(self.pos, -1, axis=0)

        # vectors (for distances a and b)
        v1 = self.pos - prev
        v2 = next - self.pos

        # side lengths
        a = np.linalg.norm(v1, axis=1)
        b = np.linalg.norm(v2, axis=1)

        # vector from prev to next (for c)
        v3 = next - prev
        c = np.linalg.norm(v3, axis=1)

        # avoid division by zero
        eps = 1e-12
        a_safe = np.maximum(a, eps)
        b_safe = np.maximum(b, eps)

        # law of cosines
        cos_theta = (a_safe**2 + b_safe**2 - c**2) / (2 * a_safe * b_safe)

        # numerical stability
        cos_theta = np.clip(cos_theta, -1.0, 1.0)

        # bending energy
        E_b = self.k_b * (1 + cos_theta)

        # boundary conditions (no angle at endpoints)
        E_b[0] = 0.0
        E_b[-1] = 0.0

        if verbose:
            print("a:", a)
            print("b:", b)
            print("c:", c)
            print("cos(theta):", cos_theta)
            print("Bending energy:", E_b)

        return E_b
        
    def calculate_energy(self, verbose=False):
        spring_energy = self.calculate_spring_energy()
        bending_energy = self.calculate_bending_energy()
        if verbose:
            print(f"spring energy:\n{spring_energy}")
            print(f"bending energy:\n{bending_energy}")
            print(f"total energy:\n{(bending_energy+spring_energy)}")
        return spring_energy + bending_energy
    
    def disturb_positions(self, s):
        noise = np.random.uniform(-s, s, size=self.pos.shape)

        # apply only to interior points
        self.pos[1:-1] += noise[1:-1]

    def monte_carlo_step(self, s, verbose=False):
        old_pos = self.pos.copy()
        old_energy = np.sum(self.calculate_energy())

        noise = np.random.uniform(-s, s, size=self.pos.shape)
        self.pos[1:-1] += noise[1:-1]

        new_energy = np.sum(self.calculate_energy())
        delta_E = new_energy - old_energy

        # if delta_E <= 0:
        #     accept = True
        # else:
        prob = np.exp(-delta_E / self.k_bT)
        accept = np.random.rand() < prob

        if not accept:
            self.pos = old_pos

        if verbose:
            print(f"ΔE = {delta_E}, accepted = {accept}")

        return accept

    def modify_l(self, eps):
        self.l = self.l * (1 + eps)

    def modify_l_task2(self, eps):
        self.l = self.l + eps

    def monte_carlo_step_1(self, s, verbose=False):
            N = self.pos.shape[0]
            if N <= 2:
                return False # Cannot perturb interior of a 2-particle system
                
            # 1. Randomly select ONE interior vertex
            i = np.random.randint(1, N - 1)
            
            # Helper function to calculate energy specifically tied to vertex 'index'
            def calc_local_energy(index):
                E_local = 0.0
                
                # --- 2 Springs ---
                # Spring left (index-1 to index)
                dist_left = np.linalg.norm(self.pos[index] - self.pos[index - 1])
                E_local += 0.5 * self.k * (dist_left - self.l[index - 1])**2
                
                # Spring right (index to index+1)
                dist_right = np.linalg.norm(self.pos[index + 1] - self.pos[index])
                E_local += 0.5 * self.k * (dist_right - self.l[index])**2
                
                # --- 3 Bending Angles ---
                # Vertices defining angles can be centered at index-1, index, and index+1 
                # (Loop bounded by the array limits to avoid endpoints where E_b = 0)
                for j in range(max(1, index - 1), min(N - 1, index + 2)):
                    a = np.linalg.norm(self.pos[j] - self.pos[j - 1])
                    b = np.linalg.norm(self.pos[j + 1] - self.pos[j])
                    c = np.linalg.norm(self.pos[j + 1] - self.pos[j - 1])
                    
                    # Prevent division by zero
                    a = max(a, 1e-12)
                    b = max(b, 1e-12)
                    
                    cos_theta = (a**2 + b**2 - c**2) / (2 * a * b)
                    cos_theta = np.clip(cos_theta, -1.0, 1.0)
                    
                    E_local += self.k_b * (1 + cos_theta)
                    
                return E_local

            # 2. Calculate local energy BEFORE the move
            E_old = calc_local_energy(i)
            
            # 3. Save original position and apply random displacement to vertex i
            old_pos_i = self.pos[i].copy()
            self.pos[i] += np.random.uniform(-s, s, size=2)
            
            # 4. Calculate local energy AFTER the move
            E_new = calc_local_energy(i)
            
            # 5. Energy Delta
            delta_E = E_new - E_old
            
            # 6. Evaluate acceptance probability
            # IF delta_E <= 0, prob > 1, so np.random.rand() < prob is always True.
            prob = np.exp(-delta_E / self.k_bT)
            accept = np.random.rand() < prob
            
            # 7. Revert position if rejected
            if not accept:
                self.pos[i] = old_pos_i
                
            if verbose:
                print(f"Vertex {i} moved. ΔE = {delta_E:.4e}, accepted = {accept}")
                
            return accept

def run_simulation(l_0, N, k, k_b, k_bT, steps, s, eps):
    system = System(l_0, N, k, k_b, k_bT)
    system.modify_l(eps)

    acceptance_buffer = []

    for i in range(steps):
        accepted = system.monte_carlo_step_1(s=s, verbose=False)

        acceptance_buffer.append(accepted)

        # keep only last 100 steps
        if len(acceptance_buffer) > 100:
            acceptance_buffer.pop(0)

        # print every 100 steps
        if (i + 1) % 1000 == 0:
            avg_acceptance = np.mean(acceptance_buffer)
            print(f"Step {i+1}: acceptance rate (last 100) = {avg_acceptance:.3f}")

    system.plot()

def run_simulation_task2(l_0, N, k, k_b, k_bT, steps, s, eps):
    system = System(l_0, N, k, k_b, k_bT)

    acceptance_buffer = []

    for i in range(steps):
        accepted = system.monte_carlo_step_1(s=s, verbose=False)

        acceptance_buffer.append(accepted)

        # keep only last 100 steps
        if len(acceptance_buffer) > 100:
            acceptance_buffer.pop(0)

        # print every 100 steps
        if (i + 1) % 1000 == 0:
            system.modify_l_task2(eps)
            print(np.average(system.l))
            system.plot()
            avg_acceptance = np.mean(acceptance_buffer)
            print(f"Step {i+1}: acceptance rate (last 100) = {avg_acceptance:.3f}")

    system.plot()

run_simulation(1.0, 3*1e2, 1e3, 1e2, 1, 10000, 0.05, 0.01)
# run_simulation_task2(1.0, 100, 1e3, 1e2, 1e-1, 10000, 0.001, 0.1)